# Week 3 — Interview Questions & Concepts

This notebook is your preparation sheet for the third week of the module. It collects ten classic Python interview questions and works through each one with a plain-language explanation and runnable code you can execute as you read. The first three questions probe the object model directly: what **decorators** are and how they wrap functions, how to check inheritance with `issubclass()`, and what Python's **Method Resolution Order (MRO)** means when a class inherits from several parents. Questions four and five tighten up naming and design choices: the meanings of single and double underscores in variable and method names, and the difference between **object-oriented** and **structural programming**. Questions six through eight cover day-to-day toolkit items — calling a base-class method without an instance, the limitations of inheritance, and how Python 3's `range()` differs from Python 2's `xrange()`. The final two questions round out the OOP essentials: overriding how objects print with `__str__` and `__repr__`, and the precise differences between instance methods, class methods, and static methods.

The questions are deliberately ordered so that each builds on the previous one. The decorator question (Q1) introduces closures and wrappers; the underscore question (Q4) explains how Python names are mangled and confers; the `__str__`/`__repr__` question (Q9) and the three-method-types question (Q10) return to behaviour that every class you write will use. Use the notebook the way an interview expects you to think: read the question, form your own answer out loud first, then compare it with the explanation. Run each code cell to confirm the claimed output — for example, in Q3 you should see the Father's method win because `Father` appears first in the inheritance list, and in Q1 you should see the decorator swap arguments before dividing. What matters is not memorising answers but being able to *derive* them: understand why the code produces what it does, and you can answer any variation on the same idea during an interview.


###`1. What is a decorator in Python?`



Python offers a unique feature called decorators.

Let's start with an analogy before getting to the technical definition of the decorators. When we mention the word "decorator", what enters your mind? Well, likely something that adds beauty to an existing object. An example is when we hang a picture frame to a wall to enhance the room.

Decorators in Python add some feature or functionality to an existing function without altering it.

Let's say we have the following simple function that takes two numbers as parameters and divides them.


```
def divide(first, second):
    print ("The result is:", first/second)
```
Now if we call this function by passing the two values 16 and 4, it will return the following output:


```
divide(16, 4)
```
The output is:

`The result is: 4.0`



### Q1 — Decorators: the one-paragraph answer

A **decorator** is a function that takes another function and returns an enhanced version of it. In the example, `swipe_decorator(func)` creates a nested `swipe(first, second)` that checks the argument order, swaps if needed, and then calls the original `func(first, second)`. Assigning `divide = swipe_decorator(divide)` replaces the name `divide` with the wrapped version — the original body is untouched, its behaviour is enriched. The `@swipe_decorator` syntax is just shorthand for exactly that assignment.

The two things to notice when the code runs:
1. `divide(4, 16)` *before* decoration gives `0.25` — the smaller number was divided by the larger.
2. After `divide = swipe_decorator(divide)`, the same call prints `4.0`, because the wrapper swapped the arguments for us.

This is the core interview demonstration of closures: the inner `swipe` function remembers `func` from the enclosing scope. It is the foundation of logging, authentication, timing, and caching decorators in production frameworks.

In [ ]:
def divide(first, second):
    print ("The result is:", first/second)

divide(16, 4)


The result is: 4.0


What will happen if we pass the number 4 first, and 16 after? The answer will be 0.25. But we don't want it to happen. We want a scenario where if we see that first < second, we swap the numbers and divide them. But we aren't allowed to change the function.

Let's create a decorator that will take the function as a parameter. This decorator will add the swipe functionality to our function.

```
def swipe_decorator(func):
    def swipe(first, second):
        if first < second:
            first, second = second, first
        return func(first, second)

    return swipe
```

Now we have generated a decorator for the divide() function. Let's see how it works.

```
divide = swipe_decorator(divide)
divide(4, 16)
```
The output is:
```
The result is: 4.0
```

We have passed the function as a parameter to the decorator. The decorator "swiped our values" and returned the function with swiped values. After that, we invoked the returned function to generate the output as expected.



In [32]:
# Func refrencinf this divide function
def divide(first, second):
    print ("The result is:", first/second)

def swipe_decorator(func):
    def swipe(first, second):
        if first < second:
            first, second = second, first
        return func(first, second)
    return swipe


divide(4, 16) # Undecorated

divide = swipe_decorator(divide) # Decorated

# Decorated
divide(4, 16)

The result is: 0.25
The result is: 4.0


In [2]:
def divide(first, second):
    print ("The result is:", first/second)

divide(4,10) # Bigger/smaller

The result is: 0.4


In [7]:
# Another way of doing the same thing

def swipe_decorator(func):
    def swipe(first, second):
        if first < second:
            first, second = second, first
        return func(first, second)
    return swipe

@swipe_decorator # divide = swipe_decorator(divide) 
def divide(first, second):
    print ("The result is:", first/second)



divide(4, 16)

The result is: 4.0


###`2. How can you determine whether a class is a subclass of another class?`



Ans: 

This is accomplished by utilising a Python function called issubclass(). The function returns true or false depending on if a class is a child of another class, indicating whether it is.

### Q2 — Checking inheritance with issubclass

`issubclass(child, parent)` returns `True` when the first argument is a subclass of the second (directly or indirectly), and `False` otherwise. In the code, `class PQR(ABC)` makes `PQR` the child of `ABC`, so `issubclass(PQR, ABC)` is `True` while `issubclass(ABC, PQR)` is `False` — the arguments order matters and the answer is not symmetrical. Distinguish it from `isinstance(obj, cls)`, which checks an *object* against a class; `issubclass` compares two *classes*. Together they are the standard runtime way to verify relationships that the type checker cannot prove statically.

In [ ]:
class ABC:
    pass

class PQR(ABC):
    pass

print(issubclass(ABC, PQR)) # False as ABC is not the child class of PQR
print(issubclass(PQR, ABC)) # True as PQR is a child os ABC

False
True


###`3.What does Python's MRO (Method Resolution Order) mean?`




Ans:

Method Resolution Order is referred to as MRO.
A class inherits from many classes under multiple inheritance.
If we attempt to access a method by building an object from the child class, the methods of the child class are first searched for the method.
If the method is not found in the child class, the inheritance classes are searched from left to right.

The `show` method is present in both the `Father` and `Mother` classes in the example presented below.

In MRO, methods and variables are searched from left to right because while conducting inheritance, `Father` class is written first and `Mother` class is written afterwards. So firstly `Father` class will be searched for `show` method if found then will get executed if not, `Mother` class will be searched.

### Q3 — Method Resolution Order in one minute

When a class inherits from several parents (multiple inheritance), Python must decide which definition of a method wins. **MRO** — Method Resolution Order — is the documented, left-to-right search order: Python looks at the child first, then walks the base classes in the order they were listed in the class definition.

In the example, `Son(Father, Mother)` lists `Father` first, so when `son.show()` is called and `Son` has no `show` of its own, Python finds `Father.show` and executes it — the output is `Father Class instance Method`, never `Mother`'s. The same rule applies to constructors: `Son.__init__` does not call `super()`, so only the Son constructor runs, but a plain attribute access follows the same left-to-right trail. You can inspect the exact order at runtime with `Son.__mro__`, which lists every class from the child up to `object` — the interviewer's favourite follow-up question.

In [ ]:
# Example code
class Father:
  def __init__(self):
    print('You are in Father Class Constructor')
   
  def show(self):
    print("Father Class instance Method")
         
class Mother:
  def __init__(self):
    print("You are in Mother Class Constructor")
   
  def show(self):
    print("Mother Class instance Method")
         
class Son(Father, Mother):
    def __init__(self):
        print("You are in Son Class Constructor")
     
son = Son()
son.show()

You are in Son Class Constructor
Father Class instance Method


###`4. What’s the meaning of single and double underscores in Python variable and method names`

* Single Leading Underscore: `_var`
* Single Trailing Underscore: `var_`
* Double Leading Underscore: `__var`
* Double Leading and Trailing Underscore: `__var__`
* Single Underscore: `_`

1. **Single Leading Underscore:** `_var` are a Python naming convention that indicates a name is meant for internal use. It is generally not enforced by the
Python interpreter and is only meant as a hint to the programmer.
 
 Adding a single underscore in front of a variable name is more like someone putting up a tiny underscore warning sign that says: 

> “Hey, this isn’t really meant to be a part of the public interface of this class. Best to leave it alone.”

### Q4 — Underscores: a quick reference

The underscore variations are one of Python's most-tested naming topics. Here they are at a glance:

| Pattern | Meaning |
|---|---|
| `_var` | single leading: "internal use" — **convention only**, not enforced |
| `var_` | single trailing: avoid clashing with a keyword (`class_`) |
| `__var` | double leading: **name mangling** — the interpreter rewrites `__baz` into `_Class__baz` |
| `__var__` | dunders: reserved magic methods like `__init__`, `__str__`, `__call__` |
| `_` | throwaway name for values you do not care about |

The code below shows `t.foo` and `t._bar` printing normally, while `self.__baz` disappears from `dir(t)` — the attribute that comes back is `_Test__baz`. That renaming protects the attribute from being accidentally overridden in a subclass. None of these except genuine dunders are enforcements: they are signals Python programmers all agree to read.

In [ ]:
class Test:
    def __init__(self):
        self.foo = 11
        self._bar = 23

t = Test()
print(t.foo) #Print 11

print(t._bar) # Print 23

11
23


2. **Single Trailing Underscore:** `var_`
 Sometimes the most fitting name for a variable is already taken by a keyword in the Python language. Therefore, names like class or def cannot be used as variable names in Python. In this case, you can append a single underscore to break the naming conflict:



In [ ]:
def make_object(name, class):
    pass

SyntaxError: ignored

In [ ]:
def make_object(name, class_):
    pass

In summary, a single trailing underscore (postfix) is used by convention to avoid naming conflicts with Python keywords. This convention
is defined and explained in PEP 8.

3. **Double Leading Underscore**: `__var`

A double underscore prefix causes the Python interpreter to rewrite
the attribute name in order to avoid naming conflicts in subclasses.

This is also called *name mangling*—the interpreter changes the name
of the variable in a way that makes it harder to create collisions when
the class is extended later.

In [ ]:
class Test:
    def __init__(self):
        self.foo = 11
        self._bar = 23
        self.__baz = 23
    
t = Test()
print(dir(t)) # This gives us a list with the object’s attributes

['_Test__baz', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_bar', 'foo']


```
['_Test__baz', '__class__', '__delattr__', '__dict__',
'__dir__', '__doc__', '__eq__', '__format__', '__ge__',
'__getattribute__', '__gt__', '__hash__', '__init__',
'__le__', '__lt__', '__module__', '__ne__', '__new__',
'__reduce__', '__reduce_ex__', '__repr__',
'__setattr__', '__sizeof__', '__str__',
'__subclasshook__', '__weakref__', '_bar', 'foo']
```

Let’s take this list and look for our original variable names `foo`, `_bar`, and `__baz`. I promise you’ll notice some interesting changes.

First of all, the `self.foo` variable appears unmodified as `foo` in the
attribute list.

Next up, `self._bar` behaves the same way—it shows up on the class
as `_bar`. Like I explained before, the leading underscore is just a convention
in this case—a hint for the programmer.

However, with `self.__baz` things look a little different. When you
search for `__baz` in that list, you’ll see that there is no variable with
that exact name.

So what happened to `__baz`?

> If you look closely, you’ll see there’s an attribute called `_Test__baz`
on this object. This is the name mangling that the Python interpreter
applies. It does this to protect the variable from getting overridden in
subclasses.

This type of variables also explained in `Private attributes` and `methods` session.

4. **Double Leading and Trailing Underscore**: `__var__`

Double underscores `__` are often referred to as “**dunders**” in the Python
community. The reason is that double underscores appear quite often
in Python code, and to avoid fatiguing their jaw muscles, Pythonistas
often shorten “**double underscore**” to “**dunder**.”

The names that have both leading and trailing double underscores are reserved for special use in the language. This rule covers things like `__init__` for object constructors, or `__call__` to make objects callable.

These dunder methods are often referred to as magic methods.

There are many dunder methods, here are some:-

`__str__, __repr__, __call__, __add__, __sub__, __len__` etc.



5. **Single Underscore** `_`:
 Sometimes used as a name for temporary or insignificant variables (“don’t care). Also, it represents the result of the last expression in a Python REPL session.


### **`5. What is the difference between OOP and SOP?`**

| Object Oriented Programming | Structural Programming|
| :-- | :-- |
| Object-Oriented Programming is a type of programming which is based on objects rather than just functions and procedures | Provides logical structure to a program where programs are divided functions |
| Bottom-up approach | Top-down approach |
| Provides data hiding | Does not provide data hiding |
| Can solve problems of any complexity | Can solve moderate problems |
| Code can be reused thereby reducing redundancy | Does not support code reusability |

### **`6. Can you call the base class method without creating an instance?`**

Yes, you can call the base class without instantiating it if:
- It is a static method
- The base class is inherited by some other subclass

### **`7. What are the limitations of inheritance?`**

- Increases the time and effort required to execute a program as it requires jumping back and forth between different classes.
- The parent class and the child class get tightly coupled.
- Any modifications to the program would require changes both in the parent as well as the child class
- Needs careful implementation else would lead to incorrect results

### **`8. What is the difference between range() and xrange()?`**

- range() creates a static list that can be iterated through while checking some conditions. This is a function that returns a list with integer sequences.
- xrange() is same in functionality as range() but it does not return a list, instead it returns an object of xrange(). xrange() is used in generators for yielding.

| range() | xrange() |
| :-- | :-- |
| In Python 3, xrange() is not supported; instead, the range() function is used to iterate in for loops. | The xrange() function is used in Python 2 to iterate in for loops.
| It returns a list. | It returns a generator object as it doesn’t really generate a static list at the run time. |
| It takes more memory as it keeps the entire list of iterating numbers in memory. | It takes less memory as it keeps only one number at a time in memory. |

## Q9 — Overriding how objects print

When Python prints an object, it asks the object to describe itself; how it does so is controlled by two special methods. Deciding which to override (and how) is a classic interview question because every class in a real system eventually wants a decent printed form.

### `__str__` — for users

- Returned when you call `str(obj)`, `f"{obj}"`, or `print(obj)`.
- Aimed at human readers: concise, friendly, readable.
- If `__str__` is missing, Python falls back on `__repr__`.

### `__repr__` — for developers

- Returned by `repr(obj)` and by evaluating the object in a REPL (the notebook cell shows the object itself uses `__repr__`).
- Aimed at unambiguous, exact *representation* — ideally a string that could rebuild the object.
- The fallback every object inherits is `<__main__.Person object at 0x...>` — the memory address, which tells you almost nothing.

The `Person` example below overrides both to return `"John Doe (30)"`, so `print(person)` and the bare-expression display both show something meaningful. Good `__repr__` is descriptive for debugging; good `__str__` is pleasant for end users. If you write only one, make it `__repr__`, because `str()` degrades gracefully to it, and it powers the debugging tools.

### Q9 in code — watch the display

The `Person` class below implements both `__str__` and `__repr__` to return `"John Doe (30)"`. Run the cell and note *which* method was invoked: a bare expression at the end of a cell (just `person`) uses `__repr__` and displays the result; `print(person)` would use `__str__` instead. Because both return the same string here you cannot tell them apart from the output alone — so the real interview lesson is understanding when each is called. In a debugger or REPL session you see `__repr__`; in user-facing `print()` statements you see `__str__`. Choose your implementation accordingly.

In [ ]:
class Person:
    def __init__(self, first_name, last_name, age):
        self.first_name = first_name
        self.last_name = last_name
        self.age = age

    def __str__(self):
        return f"{self.first_name} {self.last_name} ({self.age})"
				
    def __repr__(self):
        return f"{self.first_name} {self.last_name} ({self.age})"

person = Person("John", "Doe", 30) # thanks to __str__
person

John Doe (30)

## Q10 — instance, class, and static methods

The difference between the three method kinds is determined by what the first argument receives. The notebook class `MyClass` demonstrates all three, and its methods return tuples so you can literally see what was passed.

| Method | Decorator | First argument | Can touch |
|---|---|---|---|
| Instance | none | `self` (the object) | object state, class state via `self.__class__` |
| Class | `@classmethod` | `cls` (the class) | class state only |
| Static | `@staticmethod` | nothing implicit | neither — like a plain function in the class namespace |

Key observations from the code below:

- `obj.method()` passes the instance automatically: `self` is exactly `obj`, confirmed by the tuple `('instance method called', obj)`. The dot syntax is sugar for `MyClass.method(obj)`, which the notebook calls explicitly with the same result.
- `obj.classmethod()` receives the *class*, not the instance: `('class method called', <class '__main__.MyClass'>)`. Calling `MyClass.classmethod()` works directly on the class.
- `obj.staticmethod()` receives nothing, because Python simply does not inject `self` or `cls` — the call shows `'static method called'`.
- Calling `MyClass.method()` with no instance fails with a `TypeError` (there is no object to fill `self`), while the class and static versions work fine on the class itself.

Naming `self` and `cls` is convention only — being first in the parameter list is what matters.

Let’s begin by writing a (Python 3) class that contains simple examples
for all three method types:
```
class MyClass:

    def method(self):
        return 'instance method called', self

    @classmethod
    def classmethod(cls):
        return 'class method called', cls

    @staticmethod
    def staticmethod():
        return 'static method called'
```

**`Instance Methods`**

The first method on `MyClass`, called `method`, is a regular instance method. That’s the basic, no-frills method type you’ll use most of the time. You can see the method takes one parameter, self, which points to an instance of MyClass when the method is called. But of course, instance methods can accept more than just one parameter.

Through the self parameter, instance methods can freely access attributes and other methods on the same object. This gives them a lot of power when it comes to modifying an object’s state.

Not only can they modify object state, instance methods can also access the class itself through the `self.__class__` attribute. This means instance methods can also modify class state.  This makes instance methods powerful in terms of access restrictions—they can freely modify state on the object instance and on the class itself.




**`Class Methods`**

Let’s compare that to the second method, `MyClass.classmethod`. I marked this method with a [`@classmethod`](https://docs.python.org/3/library/functions.html#classmethod) decorator to flag it as a class method. Instead of accepting a self parameter, class methods take a `cls` parameter that points to the class—and *not* the object instance—when the method is called.

Since the class method only has access to this `cls` argument, it can’t modify object instance state. That would require access to self. However, class methods can still modify class state that applies across all instances of the class.

`Static Methods`

The third method, `MyClass.staticmethod` was marked with a [`@staticmethod6`](https://docs.python.org/3/library/functions.html#staticmethod) decorator to flag it as a static method. 

This type of method doesn’t take a `self` or a `cls` parameter, although, of course, it can be made to accept an arbitrary number of other parameters.

As a result, a static method cannot modify object state or class state.
Static methods are restricted in what data they can access—they’re primarily a way to namespace your methods.

**Let’s See Them in Action!**

Let’s take a look at how these methods behave in action when we call them. We’ll start by creating an instance of the class and then calling the three different methods on it.

`MyClass` was set up in such a way that each method’s implementation returns a tuple containing information we can use to trace what’s going on and which parts of the class or object that method can access.


### Q10 in code — watch what is passed

The class below returns tuples from every method so the output reveals what Python injected. Call `obj.method()` and the tuple shows the *instance*; call `obj.classmethod()` and it shows the *class*; call `obj.staticmethod()` and it shows a plain string with no object at all. Then try the class-level calls in the last cell: `MyClass.classmethod()` and `MyClass.staticmethod()` succeed, but `MyClass.method()` raises a `TypeError` because no instance exists to become `self`. Reading those outputs is the fastest way to internalise the difference between the three method types.

In [27]:
# Class

class MyClass:

    def method(self):
        return 'instance method called', self

    @classmethod
    def classmethod(cls):
        return 'class method called', cls

    @staticmethod
    def staticmethod():
        return 'static method called'


Here’s what happens when we call an instance method:

In [29]:
obj = MyClass()
obj.method()

('instance method called', <__main__.MyClass at 0x7faa53c48640>)

This confirms that, in this case, the instance method called method has access to the object instance (printed as `<MyClass instance>`) via the self argument.

When the method is called, Python replaces the `self` argument with the instance object, `obj`. 

We could ignore the syntactic sugar provided by the `obj.method()` **dot-call syntax** and pass the instance object manually to get the same result:
```
MyClass.method(obj)
```


In [14]:
MyClass.method(obj)

('instance method called', 10)

In [15]:
type(obj)

__main__.MyClass

Let’s try out the **class method** next:


In [22]:
obj.classmethod()

('class method called', __main__.MyClass)

Calling `classmethod()` showed us that it doesn’t have access to the `<MyClass instance>` object, but only to the `<class MyClass>` object, representing the class itself (**everything in Python is an object, even classes themselves**).

Notice how Python automatically passes the class as the first argument to the function when we call `MyClass.classmethod()`. Calling a method in Python through the **dot syntax** triggers this behavior. The `self` parameter on instance methods works the same way.

Please note that naming these parameters `self` and `cls` is just a convention. You could just as easily name them `the_object` and `the_class` and get the same result. All that matters is that they’re positioned first in the parameter list for that particular method.

Time to call the **static method** now:


In [30]:
obj.staticmethod()

'static method called'

Did you see how we called `staticmethod()` on the object and were able to do so successfully? Some developers are surprised when they learn that it’s possible to call a static method on an object instance.

Behind the scenes, Python simply enforces the access restrictions by
not passing in the `self` or the `cls` argument when a static method gets
called using the dot syntax

This confirms that static methods can neither access the object instance state nor the class state. They work like regular functions but belong to the class’ (and every instance’s) namespace.

Now, let’s take a look at what happens when we attempt to call these
methods on the class itself, **without creating an object instance** beforehand:

In [31]:
# Class Method
print(MyClass.classmethod())
# Static method
print(MyClass.staticmethod())
#Instance Method
print(MyClass.method())

('class method called', <class '__main__.MyClass'>)
static method called


TypeError: ignored

We were able to call `classmethod()` and `staticmethod(`) just fine, but attempting to call the instance method `method()` failed with a `TypeError`.

This is to be expected. This time we didn’t create an object instance and tried calling an instance function directly on the class blueprint itself. This means there is no way for Python to populate the `self` argument and therefore the call fails with a `TypeError exception`.

This should make the distinction between these three method types a
little more clear

**Key Takeaways**
* Instance methods need a class instance and can access the instance through `self`.
* Class methods don’t need a class instance. They can’t access the
instance (`self`) but they have access to the class itself via `cls`.
* Static methods don’t have access to `cls or self`. They work like
regular functions but belong to the class’ namespace.
* Static and class methods communicate and (to a certain degree)
enforce developer intent about class design. This can have definite maintenance benefits.


## Summary

This week's interview sheet covered ten pillars of Python and OOP. You saw decorators as functions that wrap other functions — closing over them to add behaviour without changing their code — and learned that `@decorator` is shorthand for `func = decorator(func)`. You met `issubclass()` for inheritance checks and the left-to-right **Method Resolution Order** that decides which parent method wins under multiple inheritance. You mapped the underscore conventions: `_var` as an unenforced convention, `var_` for keyword clashes, `__var` for name mangling, and `__var__` for magic methods. You compared OOP with structural programming, considered the costs of inheritance (coupling between parent and child), and clarified that Python 3's `range()` is a lazy generator-style object where Python 2 used the eager `xrange()`. Finally, you distinguished `__str__` (for users) from `__repr__` (for developers), and instance, class, and static methods by what their first argument receives — `self`, `cls`, or nothing. Keep the outputs in mind: they are the evidence interviewers expect you to explain.